<a href="https://colab.research.google.com/github/riccardogs/PyTorch_tutorial/blob/main/Build_the_Neural_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Neural networks comprise of layers/modules that perform operations on data. The torch.nn namespace provides all the building blocks you need to build your own neural network. Every module in PyTorch subclasses the nn.Module. A neural network is a module itself that consists of other modules (layers). This nested structure allows for building and managing complex architectures easily.

In the following sections, we’ll build a neural network to classify images in the FashionMNIST dataset.

In [21]:
import os
# import os: importa il modulo operativo di sistema
# - Serve per interagire con il file system
# - Utile per salvare modelli, caricare dati, gestire percorsi

import torch

from torch import nn
# from torch import nn: importa il modulo delle reti neurali
# - nn sta per "neural networks"
# - Contiene:
#   * Layer predefiniti: nn.Linear, nn.Conv2d, nn.LSTM, ecc.
#   * Funzioni di attivazione: nn.ReLU, nn.Sigmoid, nn.Softmax
#   * Funzioni di loss: nn.MSELoss, nn.CrossEntropyLoss
#   * Container: nn.Sequential, nn.Module (classe base per modelli)

from torch.utils.data import DataLoader
# from torch.utils.data import DataLoader: importa il caricatore di dati
# - DataLoader gestisce il caricamento in batch
# - Shuffle dei dati
# - Caricamento parallelo con più worker
# - Iterazione efficiente sui dataset

from torchvision import datasets, transforms
# from torchvision import datasets, transforms: importa moduli di torchvision
# - torchvision: pacchetto per computer vision
#
# - datasets: contiene dataset predefiniti
#   * FashionMNIST, MNIST, CIFAR10, CIFAR100, ImageNet, ecc.
#   * Scarica automaticamente i dati se non presenti
#   * Già suddivisi in training/test
#
# - transforms: contiene trasformazioni per immagini
#   * ToTensor(): converte immagini in tensor e normalizza
#   * Normalize(): normalizza con media e std dev
#   * Resize(): ridimensiona immagini
#   * RandomCrop(), RandomHorizontalFlip(): data augmentation
#   * Compose(): combina più trasformazioni

# Get Device for Training

We want to be able to train our model on an accelerator such as CUDA, MPS, MTIA, or XPU. If the current accelerator is available, we will use it. Otherwise, we use the CPU.

In [22]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
# device = ... : determina il dispositivo da usare (GPU o CPU)

# ANALISI PASSO PER PASSO:

# 1. torch.accelerator.is_available()
#    - Controlla se c'è un acceleratore disponibile (GPU NVIDIA, GPU Apple, ecc.)
#    - Restituisce True se presente, False se solo CPU

# 2. Se is_available() è True:
#    torch.accelerator.current_accelerator().type
#    - torch.accelerator.current_accelerator(): ottiene l'acceleratore corrente
#    - .type: restituisce il tipo di acceleratore come stringa
#      * "cuda" per GPU NVIDIA
#      * "mps" per GPU Apple (Metal Performance Shaders)
#      * Altri tipi possibili: "xpu", "mtia", ecc.

# 3. Se is_available() è False:
#    "cpu"  # usa la CPU come fallback

# 4. L'operatore ternario if/else in una riga:
#    valore_se_vero if condizione else valore_se_falso

print(f"Using {device} device")
# print(f"Using {device} device"): stampa il dispositivo che verrà utilizzato
# - Output esempio: "Using cuda device" (se GPU NVIDIA disponibile)
# - Output esempio: "Using mps device" (se Mac con chip M1/M2)
# - Output esempio: "Using cpu device" (se nessuna GPU disponibile)


Using cpu device


# Define the Class

We define our neural network by subclassing nn.Module, and initialize the neural network layers in __ init__. Every nn.Module subclass implements the operations on input data in the forward method.

We create an instance of NeuralNetwork, and move it to the device, and print its structure.

In [23]:
class NeuralNetwork(nn.Module):
    # class NeuralNetwork(nn.Module): definisce una nuova classe che eredita da nn.Module
    # - nn.Module è la classe base per tutti i modelli di reti neurali in PyTorch
    # - Ereditare da nn.Module ci dà accesso a funzionalità come:
    #   * Gestione automatica dei parametri
    #   * Metodi come .to(device), .train(), .eval()
    #   * Salvataggio e caricamento con .state_dict()


    def __init__(self):
        # __init__: costruttore della classe, definisce l'architettura del modello
        # Qui dichiariamo TUTTI i layer che useremo

        super().__init__()
        # super().__init__(): chiama il costruttore della classe padre (nn.Module)
        # Necessario per inizializzare correttamente il modulo PyTorch

        self.flatten = nn.Flatten()
        # self.flatten = nn.Flatten(): layer che appiattisce le immagini
        # - Input: tensor di forma (batch_size, 1, 28, 28)
        # - Output: tensor di forma (batch_size, 28*28 = 784)
        # - Trasforma l'immagine 2D in un vettore 1D per i layer lineari

        self.linear_relu_stack = nn.Sequential(
            # nn.Sequential(): contenitore che applica i layer in sequenza
            # L'output di un layer diventa input del successivo

            nn.Linear(28*28, 512),
            # Primo layer lineare (fully connected)
            # - 28*28 = 784: dimensione dell'input (pixel dell'immagine appiattita)
            # - 512: dimensione dell'output (numero di neuroni)
            # - Ogni neurone è connesso a tutti i 784 pixel di input

            nn.ReLU(),
            # Funzione di attivazione ReLU (Rectified Linear Unit)
            # - Formula: f(x) = max(0, x)
            # - Introduce non-linearità nel modello
            # - Aiuta la rete a imparare relazioni complesse

               # ESEMPIO CONCRETO:
               # Immaginiamo che il layer lineare produca questi 5 valori (in realtà sono 512):
               # output_lineare = [-2.5, 1.3, -0.7, 4.2, -3.1]

               # Dopo ReLU:
               # output_relu = [0.0, 1.3, 0.0, 4.2, 0.0]
               # - -2.5 → 0.0 (negativo diventa zero)
               # - 1.3 → 1.3 (positivo rimane)
               # - -0.7 → 0.0 (negativo diventa zero)
               # - 4.2 → 4.2 (positivo rimane)
               # - -3.1 → 0.0 (negativo diventa zero)

               # VISUALIZZAZIONE GRAFICA:
               #    ↑
               # 4  |        ●
               # 3  |
               # 2  |
               # 1  |     ●
               # 0  |  ●     ●     ●
               #    +----------------→
               #     -2  -1   0   1   2   3   4
               #
               # La ReLU "spegne" i neuroni con output negativo
               # Questo introduce non-linearità e aiuta la rete a imparare pattern complessi

            nn.Linear(512, 512),
            # Secondo layer lineare
            # - Input: 512 (dal layer precedente)
            # - Output: 512 (ancora 512 neuroni)
            # - Layer nascosto (hidden layer)

            # PERCHÉ SI CHIAMANO "NASCOSTI" (HIDDEN):
            # - Non sono direttamente accessibili dall'esterno
            # - Non vediamo direttamente cosa rappresentano
            # - L'utente vede solo input (immagini) e output (classi)
            # - I layer nascosti sono "scatole nere" che trasformano i dati internamente

            nn.ReLU(),
            # Ancora una ReLU dopo il secondo layer

            nn.Linear(512, 10),
            # Terzo layer lineare (layer di output)
            # - Input: 512
            # - Output: 10 (numero di classi in Fashion-MNIST)
            # - Produce i "logits" (punteggi grezzi per ogni classe)
        )

    def forward(self, x):
        # forward: definisce il passaggio in avanti (forward pass)
        # - È il cuore della rete neurale: descrive come i dati fluiscono attraverso il modello
        # - Questo metodo VIENE CHIAMATO AUTOMATICAMENTE quando fai model(x)
        # - NON devi chiamarlo direttamente come model.forward(x) (anche se tecnicamente funziona)

        # Parametro x:
        # - x è il tensor di input (tipicamente un batch di immagini)
        # - Shape di x: (batch_size, canali, altezza, larghezza)
        # - Per Fashion-MNIST: (64, 1, 28, 28) con batch_size=64
        # - 64 immagini in scala di grigi (1 canale) di 28x28 pixel

        x = self.flatten(x)
        # x = self.flatten(x): appiattisce le immagini 2D in vettori 1D
        # - self.flatten è un layer nn.Flatten() definito in __init__
        # - COSA FA: prende ogni immagine e la "stende" in una lunga riga
        # - Prima: (64, 1, 28, 28) → 64 immagini, ognuna è una matrice 28x28
        # - Dopo: (64, 784) → 64 immagini, ognuna è un vettore di 784 numeri
        # - 784 = 28 * 28 (tutti i pixel in fila)
        # - Perché? I layer lineari (Linear) lavorano con vettori, non con matrici 2D

        logits = self.linear_relu_stack(x)
        # logits = self.linear_relu_stack(x): passa attraverso i layer sequenziali
        # - self.linear_relu_stack è il contenitore nn.Sequential definito in __init__
        # - Contiene: Linear(784,512) → ReLU → Linear(512,512) → ReLU → Linear(512,10)

        # COSA SUCCEDE QUI (passo dopo passo):
        # 1. Primo Linear(784,512):
        #    - Input: (64, 784)
        #    - Output: (64, 512) - 512 valori per ogni immagine
        #    - Ogni valore è una combinazione lineare dei 784 pixel

        # 2. ReLU:
        #    - Input: (64, 512)
        #    - Output: (64, 512) - stessa forma, ma valori negativi azzerati
        #    - max(0, x) applicato a ogni singolo valore

        # 3. Secondo Linear(512,512):
        #    - Input: (64, 512) (dopo ReLU)
        #    - Output: (64, 512) - trasformazione lineare

        # 4. ReLU:
        #    - Input: (64, 512)
        #    - Output: (64, 512) - ancora attivazione

        # 5. Terzo Linear(512,10):
        #    - Input: (64, 512)
        #    - Output: (64, 10) - 10 valori per ogni immagine (uno per classe)

        # Il risultato finale sono i "logits":
        # - logits è un tensor di forma (batch_size, 10)
        # - Per ogni immagine, abbiamo 10 numeri (uno per ogni classe: T-shirt, Trouser, ...)
        # - Questi numeri sono detti "logits" o "punteggi grezzi"
        # - Possono essere:
        #   * Positivi o negativi
        #   * Grandi o piccoli
        #   * Non sono ancora probabilità (non sommano a 1)

        return logits
        # return logits: restituisce i logits al chiamante

        # IMPORTANTE: NON applichiamo softmax qui!
        # Perché non usiamo softmax nell'forward?

        # 1. La funzione di loss nn.CrossEntropyLoss in PyTorch:
        #    - Include già internamente la softmax
        #    - Si aspetta logits, non probabilità
        #    - Se applicassimo softmax qui, faremmo softmax due volte

        # 2. Stabilità numerica:
        #    - La softmax può causare problemi con valori estremi
        #    - CrossEntropyLoss usa trucchi numerici che funzionano meglio con logits

        # 3. Flessibilità:
        #    - A volte vuoi i logits per altri scopi (es. calcolare metriche diverse)
        #    - Puoi sempre applicare softmax dopo se necessario:
        #      probabilità = torch.softmax(logits, dim=1)

        # COSA RESTITUISCE forward():
        # Per batch_size=64: tensor di forma (64, 10)
        # Esempio per una singola immagine:
        # [2.1, -1.3, 0.5, 3.2, -0.8, 1.7, -2.1, 0.3, -0.5, 1.2]
        # Il valore più alto (3.2) corrisponde alla classe predetta (es. classe 3 = Dress)

In [24]:
model = NeuralNetwork().to(device)
# model = NeuralNetwork().to(device): crea un'istanza del modello e la sposta sul dispositivo scelto

# ANALIZZIAMO IN DUE PARTI:

# 1. NeuralNetwork()
#    - Crea un'istanza della classe NeuralNetwork che abbiamo definito prima
#    - Viene chiamato il costruttore __init__ che:
#      * Inizializza self.flatten = nn.Flatten()
#      * Inizializza self.linear_relu_stack = nn.Sequential(...)
#      * Prepara tutti i layer e i parametri (pesi e bias)
#    - I pesi sono inizializzati automaticamente (default di PyTorch)

# 2. .to(device)
#    - Sposta TUTTO il modello sul dispositivo specificato (es. "cuda", "mps", "cpu")
#    - Sposta:
#      * Tutti i parametri del modello (pesi e bias)
#      * I buffer (es. running_mean in BatchNorm)
#    - Se device = "cuda", tutto va sulla GPU
#    - Se device = "cpu", tutto rimane sulla CPU
#    - Importante: se non fai .to(device), il modello rimane su CPU

print(model)
# print(model): stampa una rappresentazione testuale dell'architettura del modello

# Output tipico:
# NeuralNetwork(
#   (flatten): Flatten(start_dim=1, end_dim=-1)
#   (linear_relu_stack): Sequential(
#     (0): Linear(in_features=784, out_features=512, bias=True)
#     (1): ReLU()
#     (2): Linear(in_features=512, out_features=512, bias=True)
#     (3): ReLU()
#     (4): Linear(in_features=512, out_features=10, bias=True)
#   )
# )

# COSA CI DICE QUESTO OUTPUT:
# - Mostra tutti i layer del modello con i loro nomi
# - Per i layer Linear:
#   * in_features: dimensione input (784 pixel, 512 neuroni, ecc.)
#   * out_features: dimensione output (512, 512, 10)
#   * bias=True: c'è un termine di bias addestrabile
# - La struttura sequenziale è chiara con i numeri (0), (1), (2), ...

# COSA CONTIENE ORA model:
# - model.flatten: il layer per appiattire le immagini
# - model.linear_relu_stack: il contenitore sequenziale
# - model.parameters(): tutti i pesi e bias del modello
#   * Circa 669.194 parametri addestrabili
# - model.to(device): assicura che tutto sia sul dispositivo giusto

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


To use the model, we pass it the input data. This executes the model’s forward, along with some background operations. Do not call model.forward() directly!

Calling the model on the input returns a 2-dimensional tensor with dim=0 corresponding to each output of 10 raw predicted values for each class, and dim=1 corresponding to the individual values of each output. We get the prediction probabilities by passing it through an instance of the nn.Softmax module.

In [25]:
X = torch.rand(1, 28, 28, device=device)
# X = torch.rand(1, 28, 28, device=device): crea un tensor casuale simulando un'immagine
# - torch.rand(): genera valori casuali uniformi tra 0 e 1
# - (1, 28, 28): forma del tensor
#   * 1: una sola immagine (batch_size=1)
#   * 28: altezza in pixel
#   * 28: larghezza in pixel
# - NOTA: manca la dimensione del canale! Dovrebbe essere (1, 1, 28, 28) per Fashion-MNIST
#   * Il modello si aspetta (batch, canali, altezza, larghezza)
#   * Qui abbiamo solo (batch, altezza, larghezza) - potrebbe funzionare lo stesso per broadcasting?
# - device=device: crea il tensor direttamente sul dispositivo scelto (GPU/CPU)

logits = model(X)
# logits = model(X): passaggio in avanti (forward pass)
# - model(X) chiama automaticamente il metodo forward (che dice come procedere nella rete) della nostra NeuralNetwork
# - X passa attraverso:
#   * flatten: da (1, 28, 28) → (1, 784) se X è (1,28,28) senza canale
#   * linear_relu_stack: trasformazioni non lineari
# - Output logits: tensor di forma (1, 10) - punteggi grezzi per 10 classi
# - Una riga (per l'unica immagine) e 10 colonne (una per classe)

pred_probab = nn.Softmax(dim=1)(logits)
# pred_probab = nn.Softmax(dim=1)(logits): converte logits in probabilità
# - nn.Softmax(dim=1): crea un layer Softmax che opera sulla dimensione 1 (le classi)
# - (logits): applica Softmax ai logits
# - Softmax: trasforma i logits in probabilità che sommano a 1
#   * formula: exp(logits_i) / sum_j exp(logits_j)
#   * Garantisce che tutti i valori siano positivi e sommino a 1
# - Output: (1, 10) con valori che rappresentano probabilità per ogni classe
# - Esempio: [0.01, 0.02, 0.05, 0.70, 0.03, 0.05, 0.02, 0.04, 0.03, 0.05]
#   * La classe 3 ha probabilità 70%

y_pred = pred_probab.argmax(1)
# y_pred = pred_probab.argmax(1): trova la classe con probabilità massima
# - .argmax(1): restituisce l'indice del valore massimo lungo la dimensione 1 (le classi)
# - Per l'esempio sopra, il massimo è 0.70 alla posizione 3
# - Output: tensor([3]) - predice che l'immagine è della classe 3 (Dress)
# - Shape: (1,) - un singolo valore per l'unica immagine

print(f"Predicted class: {y_pred}")
# print(f"Predicted class: {y_pred}"): stampa la classe predetta
# - Output esempio: "Predicted class: tensor([3], device='cuda:0')"
# - Mostra l'indice della classe predetta e su quale dispositivo si trova

Predicted class: tensor([0])


# Model Layers

Let’s break down the layers in the FashionMNIST model. To illustrate it, we will take a sample minibatch of 3 images of size 28x28 and see what happens to it as we pass it through the network.

In [26]:
input_image = torch.rand(3,28,28)

# NON sono "3 strati attaccati" nel senso di livelli sovrapposti!
# Sono 3 oggetti SEPARATI messi in un "contenitore"

# COSA RAPPRESENTA:
# - Dimensione 0 (n=3): numero di elementi/oggetti (batch)
# - Dimensione 1 (x=28): altezza di ogni oggetto
# - Dimensione 2 (y=28): larghezza di ogni oggetto

# VISUALIZZAZIONE MENTALE:
# Immagina di avere 3 foglietti 28x28 messi in una pila
# - NON è un blocco spesso 3 strati (come 3 fogli incollati)
# - Sono 3 foglietti SEPARATI che puoi sfogliare

print(input_image.size())

torch.Size([3, 28, 28])


# nn.Flatten

We initialize the nn.Flatten layer to convert each 2D 28x28 image into a contiguous array of 784 pixel values ( the minibatch dimension (at dim=0) is maintained).

In [27]:
flatten = nn.Flatten()
# flatten = nn.Flatten(): crea un layer di appiattimento
# - Di default, nn.Flatten() inizia ad appiattire dalla dimensione 1 (ignora la dimensione 0)
# - Lascia intatta la prima dimensione, appiattisce tutte le altre

flat_image = flatten(input_image)
# flat_image = flatten(input_image): applica il flatten all'immagine
# - input_image ha forma (3, 28, 28)
# - flatten lascia intatta la dimensione 0 (3)
# - Prende le dimensioni rimanenti (28, 28) e le appiattisce in un'unica dimensione
# - Calcolo: 28 × 28 = 784
# - Il risultato è un tensor di forma (3, 784)

print(flat_image.size())
# print(flat_image.size()): stampa la forma del tensor dopo il flatten
# Output: torch.Size([3, 784])

# COSA RAPPRESENTA (3, 784)?
# - 3: lo stesso numero della prima dimensione (3 immagini)
# - 784: tutti i pixel di ogni "foglio" 28x28 messi in fila

# OGNI RIGA È UN VETTORE DI 784 NUMERI

torch.Size([3, 784])


# nn.Linear

The linear layer is a module that applies a linear transformation on the input using its stored weights and biases.

In [28]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
# layer1 = nn.Linear(in_features=28*28, out_features=20): crea un layer lineare (fully connected)
# - nn.Linear: layer che applica una trasformazione lineare: y = xW^T + b
# - in_features=28*28=784: dimensione dell'input che questo layer si aspetta
#   * Ogni campione in input deve essere un vettore di 784 numeri
# - out_features=20: dimensione dell'output (numero di neuroni in questo layer)
#   * Il layer produrrà 20 valori per ogni campione in input
# - Cosa contiene questo layer:
#   * Pesi (W): matrice di forma (20, 784) - 20×784 = 15.680 parametri
#   * Bias (b): vettore di forma (20,) - 20 parametri
#   * Totale: 15.700 parametri addestrabili

hidden1 = layer1(flat_image)
# hidden1 = layer1(flat_image): applica il layer lineare all'input flat_image
# - flat_image: viene dal flatten precedente, ha forma (3, 784)
#   * 3: numero di campioni (3 immagini)
#   * 784: features (pixel appiattiti) per ogni campione
# - layer1 si aspetta input di forma (batch_size, in_features=784)
# - flat_image ha esattamente questa forma: (3, 784)
# - Cosa fa layer1:
#   * Per ognuno dei 3 campioni, moltiplica il vettore 784 per la matrice dei pesi (20×784)
#   * Aggiunge il bias
#   * Produce un vettore di 20 valori per ogni campione

print(hidden1.size())
# print(hidden1.size()): stampa la forma del tensor risultante
# Output: torch.Size([3, 20])
# - 3: stesso numero di campioni dell'input (batch size invariata)
# - 20: output features (ogni campione ora è rappresentato da 20 valori)

# VISUALIZZAZIONE DEL FLUSSO:
# Input: flat_image shape (3, 784)
#        [campione1: 784 valori]
#        [campione2: 784 valori]
#        [campione3: 784 valori]
#        ↓
# layer1 (784 → 20) si applica a OGNI campione
#        ↓
# Output: hidden1 shape (3, 20)
#        [campione1: 20 valori]
#        [campione2: 20 valori]
#        [campione3: 20 valori]

# COSA RAPPRESENTANO QUESTI 20 VALORI?
# Sono una nuova rappresentazione dell'immagine originale
# - Non sono più pixel, ma "caratteristiche apprese" (features)
# - Ogni neurone dei 20 si è specializzato nel riconoscere qualcosa
# - I valori possono essere positivi o negativi
# - Più alti = più attivazione per quella caratteristica

torch.Size([3, 20])


# nn.ReLU

Non-linear activations are what create the complex mappings between the model’s inputs and outputs. They are applied after linear transformations to introduce nonlinearity, helping neural networks learn a wide variety of phenomena.

In this model, we use nn.ReLU between our linear layers, but there’s other activations to introduce non-linearity in your model.

In [29]:
print(f"Before ReLU: {hidden1}\n\n")
# print(f"Before ReLU: {hidden1}\n\n"): stampa i valori prima dell'applicazione di ReLU
# - hidden1 è il risultato del layer lineare: tensor di forma (3, 20)
# - Contiene 3 campioni, ognuno con 20 valori (che possono essere positivi o negativi)
# - I valori sono i "logits" o attivazioni grezze dal layer lineare

hidden1 = nn.ReLU()(hidden1)
# hidden1 = nn.ReLU()(hidden1): applica la funzione ReLU a tutti gli elementi
# - nn.ReLU() crea un layer ReLU (Rectified Linear Unit)
# - (hidden1) applica questo layer al tensor hidden1
# - ReLU fa: f(x) = max(0, x) per OGNI singolo valore nel tensor
#   * Se il valore è positivo: rimane invariato
#   * Se il valore è negativo: diventa 0
# - La forma rimane la stessa: (3, 20)

print(f"After ReLU: {hidden1}")
# print(f"After ReLU: {hidden1}"): stampa i valori dopo ReLU
# - Ora tutti i valori negativi sono diventati 0
# - I valori positivi sono rimasti uguali

# I valori negativi (sotto lo 0) vengono "spenti" (portati a 0)
# I valori positivi (sopra lo 0) vengono mantenuti

Before ReLU: tensor([[ 0.0981,  0.0657,  0.0584,  0.1499, -0.2266,  0.5884, -0.1607,  0.3903,
          0.1318,  0.3611,  0.0365, -0.0639,  0.6704, -0.0096, -0.2039, -0.1106,
          0.0766,  0.3081, -0.3130, -0.0891],
        [ 0.0938,  0.1044,  0.0611, -0.1348,  0.0450,  0.7195, -0.3610,  0.0296,
          0.4408,  0.3641, -0.1817,  0.2130,  0.8506, -0.0493,  0.0667, -0.1373,
          0.3720,  0.0902, -0.1272, -0.3734],
        [ 0.0131,  0.2565, -0.1972, -0.1030, -0.2886,  0.9701, -0.5247,  0.0964,
          0.4286,  0.0380,  0.3267,  0.0962,  0.6432, -0.2930, -0.0571, -0.0303,
          0.0467,  0.3913, -0.2008, -0.2191]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0981, 0.0657, 0.0584, 0.1499, 0.0000, 0.5884, 0.0000, 0.3903, 0.1318,
         0.3611, 0.0365, 0.0000, 0.6704, 0.0000, 0.0000, 0.0000, 0.0766, 0.3081,
         0.0000, 0.0000],
        [0.0938, 0.1044, 0.0611, 0.0000, 0.0450, 0.7195, 0.0000, 0.0296, 0.4408,
         0.3641, 0.0000, 0.2130, 0.8506, 0.0000, 0.06

# nn.Sequential

nn.Sequential is an ordered container of modules. The data is passed through all the modules in the same order as defined. You can use sequential containers to put together a quick network like seq_modules.

In [30]:
seq_modules = nn.Sequential(
    # nn.Sequential: contenitore che applica i layer in SEQUENZA (uno dopo l'altro)
    # L'output di ogni layer diventa l'input del successivo

    flatten,
    # flatten: il layer nn.Flatten() definito prima
    # Input: (3, 28, 28) → Output: (3, 784)
    # Appiattisce ogni immagine 28x28 in un vettore di 784 pixel

    layer1,
    # layer1: il layer nn.Linear(784, 20) definito prima
    # Input: (3, 784) → Output: (3, 20)
    # Trasforma i 784 pixel in 20 caratteristiche

    nn.ReLU(),
    # nn.ReLU(): funzione di attivazione ReLU
    # Input: (3, 20) → Output: (3, 20) (stessa forma)
    # Azzera tutti i valori negativi, lascia invariati i positivi

    nn.Linear(20, 10)
    # nn.Linear(20, 10): nuovo layer lineare (non definito prima)
    # Input: (3, 20) → Output: (3, 10)
    # Trasforma le 20 caratteristiche in 10 punteggi (uno per classe)
    # Questo è il layer di output per Fashion-MNIST (10 classi)
)

# COSA CONTIENE ORA seq_modules:
# [Flatten] → [Linear(784→20)] → [ReLU] → [Linear(20→10)]

input_image = torch.rand(3,28,28)
# input_image = torch.rand(3,28,28): crea un tensor casuale 3x28x28
# - 3: numero di immagini nel batch (o 3 canali di una singola immagine)
# - 28: altezza
# - 28: larghezza
# Valori casuali tra 0 e 1

logits = seq_modules(input_image)
# logits = seq_modules(input_image): passa l'input attraverso TUTTA la sequenza
# PyTorch esegue automaticamente i layer in ordine:

# PASSO 1: flatten(input_image)
# (3,28,28) → (3,784)

# PASSO 2: layer1(risultato_passo1)
# (3,784) → (3,20)

# PASSO 3: ReLU(risultato_passo2)
# (3,20) → (3,20) (stessa forma, valori negativi azzerati)

# PASSO 4: nn.Linear(20,10)(risultato_passo3)
# (3,20) → (3,10)

# Output finale logits: tensor di forma (3, 10)
# - 3: stesso numero di input (batch size)
# - 10: punteggi per le 10 classi di Fashion-MNIST

print(logits.size())  # torch.Size([3, 10])

# VISUALIZZAZIONE DEL FLUSSO:
# input (3,28,28)
#     ↓
# [Flatten] → (3,784)
#     ↓
# [Linear 784→20] → (3,20)
#     ↓
# [ReLU] → (3,20)
#     ↓
# [Linear 20→10] → (3,10)
#     ↓
# logits (3,10)





# IN PRATICA:
# - Usa SEQUENTIAL quando hai una semplice pila di layer in ordine
# - Usa FORWARD quando hai bisogno di controllo personalizzato

torch.Size([3, 10])


# nn.Softmax

The last linear layer of the neural network returns logits - raw values in [-infty, infty] - which are passed to the nn.Softmax module. The logits are scaled to values [0, 1] representing the model’s predicted probabilities for each class. dim parameter indicates the dimension along which the values must sum to 1.

In [31]:
softmax = nn.Softmax(dim=1)
# softmax = nn.Softmax(dim=1): crea un layer Softmax
# - nn.Softmax è un layer di PyTorch che applica la funzione Softmax
# - dim=1: specifica lungo quale dimensione applicare Softmax
#   * dim=0: agisce sulle righe (diversi campioni nel batch)
#   * dim=1: agisce sulle colonne (le 10 classi per ogni campione) - QUESTO È QUELLO CHE VOGLIAMO
#   * In un tensor di forma (batch_size, 10), vogliamo che per OGNI campione le 10 probabilità sommino a 1

# COSA FA SOFTMAX:
# Trasforma un vettore di numeri reali (logits) in un vettore di probabilità che sommano a 1
# Formula: softmax(x_i) = exp(x_i) / sum_j exp(x_j)


pred_probab = softmax(logits)
# pred_probab = softmax(logits): applica Softmax ai logits
# - logits: tensor di forma (batch_size, 10) dal modello
#   * Per Fashion-MNIST, 10 punteggi grezzi (uno per classe)
#   * Possono essere positivi, negativi, grandi o piccoli
# - softmax(logits) trasforma OGNI riga (ogni campione) in probabilità
# - Output: stesso shape (batch_size, 10) ma ora:
#   * Ogni valore è compreso tra 0 e 1
#   * Per ogni riga, la somma dei 10 valori è 1 (100%)

# ESEMPIO CONCRETO:
# Supponiamo logits per un singolo campione:
# logits = [2.0, 1.0, 0.1, -1.0, 0.5, -0.5, 1.5, -2.0, 0.8, -0.3]

# Dopo softmax:
# pred_probab = [0.20, 0.12, 0.08, 0.01, 0.06, 0.02, 0.16, 0.003, 0.10, 0.027]
# Verifica: somma ≈ 1.00 (0.20+0.12+0.08+0.01+0.06+0.02+0.16+0.003+0.10+0.027 ≈ 1)

# PROPRIETÀ:
# - I valori più grandi in logits diventano probabilità più alte
# - I valori negativi diventano probabilità molto piccole (ma mai zero)
# - La distribuzione è "competitiva": aumentare una probabilità riduce le altre


# USO TIPICO:
# - Per ottenere probabilità dalle predizioni del modello
# - Per visualizzare quanto il modello è "sicuro" della sua predizione
# - Come input per metriche che richiedono probabilità

# Model Parameters

Many layers inside a neural network are parameterized, i.e. have associated weights and biases that are optimized during training. Subclassing nn.Module automatically tracks all fields defined inside your model object, and makes all parameters accessible using your model’s parameters() or named_parameters() methods.

In this example, we iterate over each parameter, and print its size and a preview of its values.

In [32]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    # for name, param in model.named_parameters(): itera su TUTTI i parametri del modello
    # - model.named_parameters(): metodo che restituisce un iteratore di tuple (nome, parametro)
    #   * Itera attraverso tutti i parametri addestrabili del modello
    #   * Include pesi (weight) e bias di TUTTI i layer
    # - name: stringa con il nome del parametro (es. "linear_relu_stack.0.weight")
    # - param: il tensor che contiene i valori del parametro

    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")
    # print(...): stampa le informazioni di ogni parametro
    # - Layer: {name}: nome del parametro (es. "linear_relu_stack.0.weight")
    #   * La struttura del nome riflette l'architettura:
    #     "linear_relu_stack" = il contenitore Sequential
    #     ".0" = primo layer nel Sequential (il Linear 784→512)
    #     ".weight" = i pesi di quel layer (non i bias)
    #
    # - Size: {param.size()}: forma del tensor dei parametri
    #   * Per i pesi di Linear: (out_features, in_features)
    #   * Per i bias di Linear: (out_features,)
    #
    # - Values : {param[:2]}: mostra i primi 2 valori del parametro
    #   * Per i pesi: param[:2] mostra le prime 2 righe della matrice dei pesi
    #   * Per i bias: param[:2] mostra i primi 2 valori del vettore bias
    #   * Utile per vedere che tipo di valori contengono (inizializzati casualmente)


# COSA CI DICE QUESTO OUTPUT:
# 1. Quanti parametri totali ha il modello
# 2. La forma di ogni layer (verifica che le dimensioni siano corrette)
# 3. I valori iniziali (di solito casuali o zero per i bias)
# 4. La struttura gerarchica del modello

# PERCHÉ È UTILE:
# - Debug: verificare che il modello sia costruito correttamente
# - Contare i parametri: sapere quanto è grande il modello
# - Verificare l'inizializzazione: controllare che i pesi non siano tutti uguali
# - Ispezionare layer specifici durante l'addestramento

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0220, -0.0259, -0.0266,  ...,  0.0264,  0.0234, -0.0011],
        [-0.0005, -0.0179, -0.0010,  ...,  0.0311, -0.0161, -0.0214]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([0.0069, 0.0337], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0070, -0.0264,  0.0345,  ..., -0.0404,  0.0194, -0.0415],
        [-0.0008,  0.0096,  0.0292,  ..., -0.0160,  0.0195,  0.0170]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | Si